# Lecția 09 - Model de proiectare pentru metacogniție


## Configurare

Acest notebook demonstrează tiparul de proiectare Metacognition folosind Microsoft Agent Framework.

**Cerințe prealabile:**
- Implementare Azure OpenAI configurată prin variabile de mediu
- Azure CLI autentificat (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ce este metacogniția?

Metacogniția este **gândirea despre gândire**. În contextul agenților AI, înseamnă construirea de agenți care pot:

- **Să se auto-reflecteze** asupra propriilor rezultate și a procesului de raționament
- **Să detecteze erori** și să se recupereze elegant în loc să eșueze în tăcere
- **Să evalueze** dacă răspunsurile lor sunt complete și utile
- **Să-și adapteze** strategia când o abordare inițială nu funcționează (de exemplu, trecând la un sistem de rezervă)

Un agent metacognitiv nu se limitează doar la a răspunde la întrebări — își monitorizează performanța și se ajustează din mers.


## Instrumente principale și de rezervă

Un tipar comun de metacogniție este **strategia de rezervă**. Agentul încearcă mai întâi un instrument principal; dacă eșuează (de ex., o eroare 404), agentul recunoaște eșecul și trece în mod transparent la un instrument de rezervă.

Aceasta reflectă sistemele din lumea reală în care serviciile principale pot fi indisponibile, iar agenții trebuie să diagnosticheze singuri problema înainte de a alege o cale alternativă.

Mai jos definim două instrumente pentru căutarea zborurilor:
- **Principal** — acoperă Paris, Tokyo și Barcelona
- **De rezervă** — acoperă Berlin, Sydney și New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Self-Reflecting Agent with Error Recovery

Agentul de mai jos este instruit să încerce mai întâi sistemul principal de zbor, să recunoască eșecurile și să treacă în mod transparent la sistemul de rezervă. După fiecare răspuns se auto-evaluează pe scurt dacă a răspuns pe deplin la întrebarea utilizatorului.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Model de autoevaluare

O altă fațetă a metacogniției este **autoevaluarea**: un agent separat (sau același agent într-o a doua trecere) revizuiește un răspuns pentru completitudine, acuratețe și utilitate.

Mai jos creăm un agent `ResponseEvaluator` care evaluează răspunsurile agenților de turism pe trei dimensiuni.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Rezumat

În această lecție ai învățat cum să construiești **agenți metacognitivi** folosind Microsoft Agent Framework:

- **Auto-reflecție**: Agenți care își monitorizează propriul raționament și comunică în mod transparent ce s-a întâmplat.
- **Recuperare a erorilor cu fallback-uri**: Un tipar cu instrument principal + de rezervă în care agentul detectează eșecurile (de exemplu, erori 404) și încearcă automat o sursă alternativă.
- **Auto-evaluare**: Un agent evaluator separat care atribuie scoruri răspunsurilor pentru completitudine, acuratețe și utilitate.

Aceste tipare fac agenții mai robusti, transparenți și de încredere — calități critice pentru implementările în producție.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Declinare de responsabilitate:
Acest document a fost tradus folosind serviciul de traducere asistată de AI Co-op Translator (https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original, în limba sa nativă, trebuie considerat sursa autorizată. Pentru informații critice, se recomandă o traducere profesională realizată de un traducător uman. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care rezultă din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
